In [3]:
spark.stop()

In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# %%
spark = SparkSession.builder \
    .appName("SECOP_RegresionLineal") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [5]:
# Cargar datos
df = spark.read.parquet("/opt/spark-data/processed/secop_features.parquet")

# Renombrar columnas para consistencia
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_raw", "features")

In [3]:
df.show()

+-----------------------+-----------+--------------------+--------------------+------------+--------------------+------------------+--------------------+---------------+----------------------+---------------------+----+----+----------------+--------------------+-------------------+----------------+--------------------+-------------------+--------------------+
|referencia_del_contrato|nit_entidad|      nombre_entidad|        departamento|      ciudad|    tipo_de_contrato|valor_del_contrato|      fecha_de_firma|estado_contrato|valor_del_contrato_num|fecha_de_firma_parsed|anio| mes|departamento_idx|tipo_de_contrato_idx|estado_contrato_idx|departamento_vec|tipo_de_contrato_vec|estado_contrato_vec|        features_raw|
+-----------------------+-----------+--------------------+--------------------+------------+--------------------+------------------+--------------------+---------------+----------------------+---------------------+----+----+----------------+--------------------+--------------

In [6]:
# Filtrar valores nulos
df = df.filter(col("label").isNotNull())
print(f"Registros: {df.count():,}")
print(f"Columnas: {len(df.columns)}")
### Pregunta: ¿Qué proporción usarías para train vs test?


Registros: 100,000
Columnas: 24


### Pregunta: ¿Qué proporción usarías para train vs test?

*Respuesta:*  
La proporción seleccionada es *B) 70/30 - Balance clásico*.

Esta proporción ofrece un equilibrio adecuado entre la cantidad de datos utilizados para entrenar el modelo y la cantidad de datos reservados para evaluar su desempeño. Permite entrenar un modelo robusto sin sacrificar la capacidad de validarlo correctamente con datos no vistos.

### Consideración: ¿Qué pasa si tienes 1 millón de registros vs 1000?

La proporción óptima depende del tamaño del dataset:

- *Con 1000 registros (dataset pequeño):*  
  Es recomendable usar proporciones como *70/30 o 80/20*, ya que se necesita una cantidad suficiente de datos para entrenar el modelo, pero también mantener un conjunto de prueba representativo.

- *Con 1 millón de registros (dataset grande):*  
  Incluso una proporción como *90/10* sería válida, ya que el 10% representaría 100,000 registros, una muestra más que suficiente para evaluar el modelo. En este caso, se prioriza maximizar la cantidad de datos para entrenamiento.

*Conclusión:*  
En este proyecto se utiliza *70/30* por ser un estándar ampliamente aceptado y adecuado al volumen de datos disponible, garantizando un buen balance entre entrenamiento y evaluación.

In [11]:
train_ratio = 0.7  # TODO: Ajusta según tu decisión
test_ratio = 0.3

train, test = df.randomSplit([train_ratio, test_ratio], seed=42)

print(f"Train: {train.count():,} registros ({train_ratio*100:.0f}%)")
print(f"Test: {test.count():,} registros ({test_ratio*100:.0f}%)")

Train: 70,007 registros (70%)


Test: 29,993 registros (30%)


¿Por qué es importante usar seed=42?

Respuesta:

Se utiliza seed=42 para garantizar la reproducibilidad del experimento. 
Al fijar la semilla, la partición de los datos en entrenamiento y prueba 
será siempre la misma en cada ejecución, permitiendo comparar resultados de forma consistente y replicable.

In [12]:
## RETO 2: Configurar el Modelo
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,           # TODO: ¿Es suficiente?
    regParam=0.0,          # Sin regularización
    elasticNetParam=0.0    # No aplica sin regParam
)

print("✓ Modelo configurado")
print(f"  maxIter: {lr.getMaxIter()}")
print(f"  regParam: {lr.getRegParam()}")


✓ Modelo configurado
  maxIter: 100
  regParam: 0.0


In [13]:
## PASO 3: Entrenar el Modelo
print("Entrenando modelo de regresión lineal...")
lr_model = lr.fit(train)

print("✓ Modelo entrenado")
print(f"  Iteraciones completadas: {lr_model.summary.totalIterations}")
print(f"  RMSE (train): ${lr_model.summary.rootMeanSquaredError:,.2f}")
print(f"  R² (train): {lr_model.summary.r2:.4f}")

Entrenando modelo de regresión lineal...


26/02/14 01:05:54 WARN Instrumentation: [0c247a22] regParam is zero, which might cause numerical instability and overfitting.
26/02/14 01:05:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/14 01:05:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/14 01:05:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/02/14 01:05:56 WARN Instrumentation: [0c247a22] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


✓ Modelo entrenado
  Iteraciones completadas: 9
  RMSE (train): $2,265,839,710.27
  R² (train): 0.0181


26/02/14 01:21:28 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/02/14 01:21:28 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce

## RETO 3: Interpretar R²
## Respuesta

*Opción correcta: B) El modelo explica 65% de la varianza en los datos.*

El coeficiente de determinación R² mide qué proporción de la variabilidad de la variable objetivo puede ser explicada por el modelo.  
Un valor de R² = 0.65 significa que el 65% de la variación en el valor real de los datos es explicada por las variables predictoras incluidas en el modelo, mientras que el 35% restante se debe a factores no modelados, ruido o relaciones no capturadas.

No significa que el modelo sea “65% preciso”, ya que R² no es una métrica de exactitud, sino de capacidad explicativa.

---

## ¿Es 0.65 un buen R²?

Depende de:

- *El dominio del problema*:  
  En problemas sociales o económicos (como contratos públicos), un R² de 0.65 es considerado *bueno*, ya que los datos suelen ser muy ruidosos y complejos.

- *La naturaleza de los datos*:  
  En datos financieros o reales, valores entre 0.3 y 0.7 ya son razonables.  
  En problemas físicos o de ingeniería, se esperan R² más altos (>0.9).

- *El objetivo del modelo*:  
  Si el objetivo es exploratorio o de apoyo a decisiones, 0.65 es bastante aceptable.  
  Si es un sistema crítico de predicción, podría requerirse un R² mayor.

In [8]:
print("Aplicando PCA")

pca_model = pca.fit(df_scaled)
df_pca = pca_model.transform(df_scaled)

df_pca.select("features_pca").first()[0]

Aplicando PCA


26/01/30 00:18:34 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


DenseVector([1.0551, 1.4962, 0.1533, 2.1719, -2.7771, 2.2736, -1.3492, -0.0368, 0.7913, -1.1381, 0.6815, -0.4323, 0.8747, -1.0002, 0.9875, 1.6206, -0.6204, -0.3367, 0.2656, 0.4688, -0.1257, -0.1813, 0.062, 0.1999, 0.0193])

In [9]:
explained_variance = pca_model.explainedVariance
sum(explained_variance)

0.5692633737728403

In [10]:
pipeline_transform = Pipeline(stages=[scaler, pca])
pipeline_transform_model=pipeline_transform.fit(df)
df_final=pipeline_transform_model.transform(df)

df_ml=df_final.select("features_pca","valor_del_contrato_num")



In [11]:
pipeline_path= "/opt/spark-data/processed/transformation_pipeline"
output_path= "/opt/spark-data/processed/secop_ml_ready.parquet"
df_ml.write.mode("overwrite").parquet(output_path)
pipeline_transform_model.save(pipeline_path)

26/01/30 00:18:36 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/01/30 00:18:37 WARN FileUtil: Failed to delete file or dir [/opt/spark-data/processed/transformation_pipeline/metadata/_temporary]: it still exists.
26/01/30 00:18:39 WARN FileUtil: Failed to delete file or dir [/opt/spark-data/processed/transformation_pipeline/stages/0_StandardScaler_dd6d85fac75e/metadata/_temporary]: it still exists.
26/01/30 00:18:40 WARN FileUtil: Failed to delete file or dir [/opt/spark-data/processed/transformation_pipeline/stages/0_StandardScaler_dd6d85fac75e/data/_temporary]: it still exists.
26/01/30 00:18:43 WARN FileUtil: Failed to delete file or dir [/opt/spark-data/processed/transformation_pipeline/stages/1_PCA_52c4b349aab5/data/_temporary]: it still exists.
